# Entrenar Modelos
En este programa se entrenarán los modelos K-NN, SVM, Naive Bayes y Random Forest. Estos tendrán un entrenamiento por cada dataset diferente generado por el archivo 'setup.ipynb, usando validación cruzada de 5 pliegues. El resultado será devuelto en el siguiente sistema de carpetas. Primero, una carpeta por modelo, dentro de cada una tendremos una carpeta por cada dataset diferente generado, y dentro de cada una de estas tendremos un archivo de modelo por cada pliegue de kfold.

# Cargamos Librerías
Cargamos las librería necesarias para este archivo

In [ ]:
import pandas as pd
import sklearn as sk
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
import joblib
import pathlib as pl
import os

# CONSTANTES
TYPES = ['original', 'estandarizado', 'normalizado']
VARIANTS = ['', '_PCA95', '_PCA80']
FOLDER_PREFIX = 'conj'
TRAINING_FILE_REGEX = 'training*.csv'
DATA_PATH = './kfolds_data'
MODEL_PATH = './trained_models'
MODEL_EXT = 'joblib'

## Función Entrenar modelo con kfold
Vamos a definir una función llamada train_model_csv() la cual acepte los siguientes parámetros:
- model: Modelo a entrenar por la función
- data_csv: Archivo csv a usar para el entrenamiento del modelo
- output_file: Ruta del archivo a devolver por el entrenamiento

In [ ]:
def train_model_csv(model, data_csv, output_file):
    # Cargamos el fragmento de entrenamiento
    data = pd.read_csv(data_csv)
    
    X = data.drop('Result', axis=1)
    Y = data['Result']
    
    # Entrenamos el modelo
    model.fit(X, Y)
    
    # Guardamos el modelo entrenado
    joblib.dump(model, output_file)

Ahora vamos a definir una funcion que dado un modelo entrene todas las variantes de datasets y sus kfolds usando train_model_csv

- Nombre: train_model
- Parámetros:
    - model: Modelo limpio a usar para entrenar
    - model_name: Nombre del modelo

In [ ]:
def train_model(model, model_name):
    print(f"Iniciando entrenamiento de: {model_name}")
    for tipo in TYPES:
        for var in VARIANTS:
            # Construimos la ruta de la carpeta
            nombre_conjunto = f"{tipo}{var}"
            ruta = pl.Path(f'{DATA_PATH}/{FOLDER_PREFIX}_{nombre_conjunto}/')
            
            # Buscamos los archivos training_1.csv, training_2.csv, etc.
            training_files = sorted([x.name for x in ruta.glob(TRAINING_FILE_REGEX)])
            
            if not training_files:
                continue

            # Creamos la carpeta de salida para los modelos si no existe
            os.makedirs(f'{MODEL_PATH}/{model_name}/{nombre_conjunto}', exist_ok=True)
            
            for train_file in training_files:
                data_csv = ruta / train_file
                
                # Definimos el nombre del archivo de salida
                fold_num = train_file.split('_')[1].split('.')[0]
                output_file = f'{MODEL_PATH}/{model_name}/{nombre_conjunto}/{model_name}{fold_num}_{nombre_conjunto}.{MODEL_EXT}'
                
                # Clonamos el modelo para empezar de cero en cada fold
                fresh_model = sk.base.clone(model)
                train_model_csv(fresh_model, data_csv, output_file)
                
            print(f"  - Completado: {nombre_conjunto}")

# Entrenamos todos los modelos
Por cada modelo correspondiente, iteramos por cada uno de los .csv y llamamos a la funcion train_model con los parametros correspondientes.

### Entrenamos KNN

In [ ]:
knn_model = KNeighborsClassifier(n_neighbors=5)
train_model(knn_model, 'KNN')

### Entrenamos SVM

In [ ]:
# Nota: SVM puede ser lento con datasets muy grandes
svm_model = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True)
train_model(svm_model, 'SVM')

### Entrenamos Naive Bayes

In [ ]:
naive_bayes_model = GaussianNB()
train_model(naive_bayes_model, 'NaiveBayes')

### Entrenamos Random Forest

In [ ]:
random_forest_model = RandomForestClassifier(n_estimators=100, random_state=42)
train_model(random_forest_model, 'RandomForest')